# Amazon Fire & Deforestation Interactive Dashboard
### Tsuquindaro Enyart

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import dcc, html, Input, Output
import requests
import warnings
warnings.filterwarnings('ignore')

### Load & Clean Data

In [2]:
fires_df = pd.read_csv("inpe_brazilian_amazon_fires_1999_2019.csv")
fires_df.columns = fires_df.columns.str.strip()
fires_df['month'] = fires_df['month'].astype(int)
fires_df['year']  = fires_df['year'].astype(int)

elnino_df = pd.read_csv("el_nino_la_nina_1999_2019.csv", encoding='utf-8-sig')
elnino_df.columns = elnino_df.columns.str.strip()

def_df = pd.read_csv("def_area_2004_2019.csv", encoding='utf-8-sig')
def_df.columns = def_df.columns.str.strip()
def_df.rename(columns={'Ano/Estados': 'year'}, inplace=True)
def_df['year'] = def_df['year'].astype(int)

# Load GeoJSON
url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"
geojson = requests.get(url).json()

In [3]:
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

state_abbr = {
    'AC':'ACRE','AM':'AMAZONAS','AP':'AMAPA',
    'MA':'MARANHAO','MT':'MATO GROSSO','PA':'PARA',
    'RO':'RONDONIA','RR':'RORAIMA','TO':'TOCANTINS'
}

def label_year(yr):
    for _, row in elnino_df.iterrows():
        if row['start year'] <= yr <= row['end year']:
            return row['phenomenon'], row['severity']
    return 'Neutral', 'Neutral'

annual_fires = fires_df.groupby('year')['firespots'].sum().reset_index()
annual_fires[['phenomenon','severity']] = annual_fires['year'].apply(
    lambda y: pd.Series(label_year(y)))

def_annual = def_df[['year','AMZ LEGAL']].rename(columns={'AMZ LEGAL':'deforestation_km2'})
merged = annual_fires.merge(def_annual, on='year', how='left')

def_states = (def_df.drop(columns=['AMZ LEGAL'])
              .melt(id_vars='year', var_name='state_abbr', value_name='deforestation_km2'))
def_states['state'] = def_states['state_abbr'].map(state_abbr)

print("annual_fires sample:")
print(annual_fires.head())
print("\nmerged nulls:", merged['deforestation_km2'].isna().sum())

annual_fires sample:
   year  firespots phenomenon  severity
0  1999      62858    La Nina    Strong
1  2000      48168    La Nina      Weak
2  2001      69675    La Nina      Weak
3  2002     273873    El Nino  Moderate
4  2003     174400    El Nino  Moderate

merged nulls: 5


### Colors and Layout

In [4]:
ALL_STATES = sorted(fires_df['state'].unique())
YEAR_MIN   = int(fires_df['year'].min())
YEAR_MAX   = int(fires_df['year'].max())
PHENOMENA  = ['All', 'El Nino', 'La Nina', 'Neutral']
INIT_YR    = [YEAR_MIN, YEAR_MAX]

COLOURS = {
    'bg':      '#01411C',
    'panel':   '#355E3B',
    'accent1': '#ddd2bf',
    'neutral': '#eceffe',
    'El Nino': '#D01F1F',
    'La Nina': '#2b9cd9',
    'Neutral': '#6a6b6b',
    'Title' :  '#FFFFFF',
}
FONT = 'Roboto Condensed, sans-serif'

card_style = {
    'background': COLOURS['panel'],
    'borderRadius': '10px',
    'padding': '10px',
    'boxShadow': '0 2px 8px rgba(0,0,0,0.4)',
}
label_style = {
    'color': COLOURS['neutral'], 'fontSize': '15px', 'fontFamily': FONT,
    'marginBottom': '5px', 'fontWeight': '600',
    'textTransform': 'uppercase', 'letterSpacing': '0.5px', 'display': 'block',
}

def base_layout(title=''):
    return dict(
        title=dict(text=title, font=dict(color='white', size=15, family=FONT)),
        paper_bgcolor=COLOURS['panel'],
        plot_bgcolor=COLOURS['bg'],
        font=dict(color=COLOURS['neutral'], family=FONT, size=15),
        margin=dict(l=50, r=20, t=45, b=45),
        xaxis=dict(gridcolor='#1e2e3d', linecolor='#1e2e3d', zerolinecolor='#1e2e3d'),
        yaxis=dict(gridcolor='#1e2e3d', linecolor='#1e2e3d', zerolinecolor='#1e2e3d'),
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(size=15)),
    )

def filter_fires(states, yr, mo=None):
    df = fires_df.copy()
    if states:
        df = df[df['state'].isin(states)]
    df = df[(df['year'] >= yr[0]) & (df['year'] <= yr[1])]
    if mo:
        df = df[(df['month'] >= mo[0]) & (df['month'] <= mo[1])]
    return df

print("Constants ready. States:", ALL_STATES)
print("card_style and label_style defined ✓")

Constants ready. States: ['ACRE', 'AMAPA', 'AMAZONAS', 'MARANHAO', 'MATO GROSSO', 'PARA', 'RONDONIA', 'RORAIMA', 'TOCANTINS']
card_style and label_style defined ✓


In [5]:
ACCENT = COLOURS['accent1']

enso_info = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px'}, children=[
    html.Label('What are El Niño & La Niña?', style={
        **label_style, 'fontSize': '16px', 'marginBottom': '12px'
    }),
    html.Div(style={'display': 'flex', 'gap': '16px', 'flexWrap': 'nowrap'}, children=[

        # El Nino box
        html.Div(style={
            'flex': '1', 'minWidth': '120px',
            'background': 'rgba((82,70,65))',
            'border': f'3px solid {COLOURS["El Nino"]}',
            'borderRadius': '8px', 'padding': '12px'
        }, children=[
            html.P('El Niño', style={'color': COLOURS['neutral'], 'fontWeight': '700',
                                         'margin': '0 0 6px', 'fontSize': '16px'}),
            html.P('Warm Pacific Ocean temperatures reduce rainfall over the Amazon, '
                   'making the forest hotter and drier, ideal conditions for fires to ignite and spread.',
                   style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.6'}),
        ]),

        # La Nina box
        html.Div(style={
            'flex': '1', 'minWidth': '120px',
            'background': 'rgba((82,70,65)',
            'border': f'3px solid {COLOURS["La Nina"]}',
            'borderRadius': '8px', 'padding': '13px'
        }, children=[
            html.P('La Niña', style={'color': COLOURS['neutral'], 'fontWeight': '700',
                                         'margin': '0 0 6px', 'fontSize': '16px'}),
            html.P('Cool Pacific Ocean temperatures bring heavier rainfall to the Amazon, '
                   'keeping vegetation wetter and making it harder for fires to start.',
                   style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.6'}),
        ]),

        # Neutral box
        html.Div(style={
            'flex': '1', 'minWidth': '120px',
            'background': 'rgba(138,155,176,0.10)',
            'border': f'3px solid {COLOURS["neutral"]}',
            'borderRadius': '8px', 'padding': '13px'
        }, children=[
            html.P('Neutral', style={'color': COLOURS['neutral'], 'fontWeight': '700',
                                        'margin': '0 0 6px', 'fontSize': '16px'}),
            html.P('Neither pattern is active. The Amazon experiences typical seasonal rainfall '
                   'with moderate fire risk depending on land use and dry season length.',
                   style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.6'}),
        ]),

    ]),
])

firespot_info = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px',
                                'borderLeft': '4px solid ' + ACCENT}, children=[
    html.Label('What is a Fire Spot?', style={
        **label_style, 'fontSize': '16px', 'marginBottom': '13px'
    }),
    html.P(
        'A fire spot is a single location where a satellite detected active fire or intense heat on the ground. '
        "Brazil's space agency INPE uses NASA satellites that continuously scan the Amazon for thermal anomalies, "
        'any pixel that is unusually hot gets flagged as a fire spot. '
        'One large fire can produce many fire spots if it burns across multiple satellite pixels, '
        'so a high fire spot count means more fires, bigger fires, or both. '
        'Fire spots are not the same as the number of individual fires, '
        'they are a measure of how much total burning activity was detected across the landscape.',
        style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.7'}
    ),
])

hotspot_info = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px',
                                'borderLeft': '4px solid ' + ACCENT}, children=[
    html.Label('Why Pará & Mato Grosso?', style={
        **label_style, 'fontSize': '16px', 'marginBottom': '13px'
    }),
    html.P('These two states sit on the "arc of deforestation" the frontier where the Amazon meets '
           'farmland. Farmers and ranchers set fires to clear land for cattle and soy, and because '
           'both states are enormous in size, they consistently record the highest fire activity.',
           style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.7'}),
])


def make_stat_panel(states, yr, mo=None):
    cards = make_stat_cards(states, yr, mo)
    ACCENT = COLOURS['accent1']
    return html.Div(
        style={'display': 'flex', 'gap': '12px', 'flexWrap': 'nowrap', 'marginBottom': '14px'},
        children=[
            html.Div(style={
                **card_style,
                'flex': '1',
                'borderTop': '3px solid ' + ACCENT,
                'textAlign': 'center',
                'padding': '14px'
            }, children=[
                html.P(title, style={'color': COLOURS['neutral'], 'fontSize': '15px',
                                     'margin': '0 0 6px', 'fontWeight': '600',
                                     'textTransform': 'uppercase', 'letterSpacing': '0.5px'}),
                html.P(value, style={'color': COLOURS['accent1'], 'fontSize': '22px',
                                     'fontWeight': '700', 'margin': '0 0 4px',
                                     'fontFamily': FONT}),
                html.P(sub, style={'color': COLOURS['neutral'], 'fontSize': '15px', 'margin': '1'}),
            ]) for title, value, sub in cards
        ]
    )

dataset_panel = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px',
                                 'borderLeft': '4px solid ' + ACCENT, 'marginTop': '14px'}, children=[
    html.Label('About the Data:', style={
        'color': COLOURS['neutral'], 'fontSize': '16px', 'fontFamily': FONT,
        'marginBottom': '12px', 'fontWeight': '600', 'display': 'block'
    }),
    html.P('The Amazon rainforest spans nine nations and covers 5.5 million km², with 60% contained '
           'within Brazil. It is home to nearly 500 indigenous communities and one of the most biodiverse '
           'ecosystems on Earth. Despite its importance, it faces constant threat from deforestation and fire, '
           'driven by illegal agriculture, logging, mining, and climate variability.',
           style={'color': COLOURS['neutral'], 'margin': '0 0 8px', 'fontSize': '15px', 'lineHeight': '1.7'}),
    html.P('Fire spot data from INPE BDQ, Deforestation data from INPE PRODES, ENSO classifications from Golden Gate Weather Services. '
           'All datasets cover 1999–2019 and are public domain.',
           style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '14px',
                  'lineHeight': '1.7',}),
])

author_panel = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '14px 20px',
                                'borderLeft': '4px solid ' + ACCENT, 'marginTop': '14px'}, children=[
    html.Label('Author: Tsuquindaro Enyart', style={
        'color': COLOURS['neutral'], 'fontSize': '16px', 'fontFamily': FONT,
        'marginBottom': '8px', 'fontWeight': '600', 'display': 'block'
    }),
    html.P('Course: Data Visualization 410   Date: March 15th, 2026',
           style={'color': COLOURS['neutral'], 'margin': '0 0 4px', 'fontSize': '15px', 'lineHeight': '1.7'}),
])

deforestation_panel = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px',
                                       'borderLeft': '4px solid ' + ACCENT}, children=[
    html.Label('Which States Lost the Most Forest?', style={
        'color': COLOURS['neutral'], 'fontSize': '16px', 'fontFamily': FONT,
        'marginBottom': '8px', 'fontWeight': '600', 'display': 'block'
    }),
    html.P('Pará lost the most forest overall, 62,778 km² between 2004 and 2019, nearly double '
           'Mato Grosso\'s 43,065 km². Most states peaked in 2004 and fell sharply after Brazil\'s '
           'deforestation action plan. However, Amazonas and Roraima tell a different story, '
           'both peaked in 2019, meaning deforestation is still accelerating there even as other '
           'states improved. Amapá and Tocantins remain relatively unaffected with the lowest totals.',
           style={'color': COLOURS['neutral'], 'margin': '0', 'fontSize': '15px', 'lineHeight': '1.7'}),
])

intro_panel = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '16px 20px',
                               'borderLeft': '4px solid ' + ACCENT}, children=[
    html.Label('Welcome!', style={
        'color': COLOURS['neutral'], 'fontSize': '16px', 'fontFamily': FONT,
        'marginBottom': '8px', 'fontWeight': '600', 'display': 'block'
    }),
    html.P('This dashboard explores fire activity and deforestation across the Brazilian Amazon '
           'between 1999 and 2019, and how climate patterns like El Niño and La Niña influence them. '
           'Use the filters to explore fire trends by year and state, and see how human '
           'land clearing and natural climate cycles combine to threaten one of the world\'s most '
           'important ecosystems.',
           style={'color': COLOURS['neutral'], 'margin': '0', 'fontSize': '15px', 'lineHeight': '1.7'}),
])

tree_panel = html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '8px',
                              'textAlign': 'center'}, children=[
    html.Img(
        src='https://www.kindpng.com/picc/m/170-1701857_plant-natural-stem-forest-flower-amazon-rainforest-tree.png',
        style={
            'width': '100%',
            'height': '150px',
            'objectFit': 'cover',
            'borderRadius': '8px',
        }
    ),
])


### Static Figures 

In [6]:
def make_fig_annual(states, yr):
    df  = filter_fires(states, yr)
    ann = df.groupby('year')['firespots'].sum().reset_index()
    ann[['phenomenon','severity']] = ann['year'].apply(lambda y: pd.Series(label_year(y)))
    fig = go.Figure()
    for phase in ['El Nino','La Nina','Neutral']:
        sub = ann[ann['phenomenon'] == phase]
        fig.add_trace(go.Bar(
            x=sub['year'], y=sub['firespots'], name=phase,
            marker_color=COLOURS[phase],
            hovertemplate='<b>%{x}</b><br>Fire spots: %{y:,}<extra>'+phase+'</extra>'
        ))
    fig.update_layout(**base_layout('Annual Fire Spots by ENSO Phase'), barmode='stack')
    fig.update_xaxes(title='Year')
    fig.update_yaxes(title='Fire Spots')
    return fig

def make_fig_seasonal(states, yr):
    df  = filter_fires(states, yr)
    mon = df.groupby('month')['firespots'].mean().reset_index()
    mon['month_name'] = mon['month'].map(month_names)
    fig = go.Figure(go.Scatter(
        x=mon['month_name'], y=mon['firespots'],
        mode='lines+markers',
        line=dict(color=COLOURS['El Nino'], width=2.5),
        marker=dict(color=COLOURS['El Nino'], size=7),
        fill='tozeroy', fillcolor='rgba(224,92,42,0.15)',
        hovertemplate='<b>%{x}</b><br>Avg fires: %{y:,.0f}<extra></extra>'
    ))
    fig.update_layout(**base_layout('Avg Monthly Fire Seasonality'))
    fig.update_xaxes(title='Month')
    fig.update_yaxes(title='Avg Fire Spots')
    return fig

def make_fig_heatmap(yr):
    df = fires_df[(fires_df['year'] >= yr[0]) & (fires_df['year'] <= yr[1])]
    pivot = (df.groupby(['state','year'])['firespots'].sum().reset_index()
               .pivot(index='state', columns='year', values='firespots').fillna(0))
    fig = go.Figure(go.Heatmap(
        z=pivot.values, x=pivot.columns.astype(str), y=pivot.index,
        colorscale='YlOrRd',
        hovertemplate='<b>%{y}</b> – %{x}<br>Fire spots: %{z:,}<extra></extra>',
        colorbar=dict(
            title=dict(text='Fires', font=dict(color='white')),
            tickfont=dict(color='white')
        )
    ))
    fig.update_layout(**base_layout('Fire Spots by State & Year'))
    fig.update_xaxes(title='Year', tickangle=-45)
    return fig

def make_fig_def_state(states, yr):
    df = def_states.copy()
    if states:
        df = df[df['state'].isin(states)]
    df = df[(df['year'] >= yr[0]) & (df['year'] <= yr[1])].dropna(subset=['state'])
    
    AMAZON_PALETTE = [
        '#e05c2a',  
        '#ddd2bf', 
        '#3ba3c9', 
        '#f0a500', 
        '#c45c8a', 
        '#7ecba1', 
        '#e8d5a3', 
        '#5b8dd9', 
        '#f28b50',  
    ]
    
    fig = go.Figure()
    for i, st in enumerate(sorted(df['state'].unique())):
        sub = df[df['state'] == st].sort_values('year')
        fig.add_trace(go.Scatter(
            x=sub['year'], y=sub['deforestation_km2'],
            name=st.title(), stackgroup='one', mode='lines',
            line=dict(width=0, color='rgba(0,0,0,0)'),
            fillcolor=AMAZON_PALETTE[i % len(AMAZON_PALETTE)],
            hovertemplate='<b>' + st.title() + '</b> %{x}<br>%{y:,} km²<extra></extra>'
        ))
    fig.update_layout(**base_layout('Deforestation by State (km²)'))
    fig.update_xaxes(title='Year')
    fig.update_yaxes(title='Deforestation (km²)')
    return fig
    
def make_fig_state_bar(yr):
    df = fires_df[(fires_df['year'] >= yr[0]) & (fires_df['year'] <= yr[1])].copy()
    agg = df.groupby('state')['firespots'].sum().sort_values(ascending=True).reset_index()
    fig = go.Figure(go.Bar(
        x=agg['firespots'], y=agg['state'], orientation='h',
        marker=dict(color=agg['firespots'], colorscale='YlOrRd', showscale=False),
        hovertemplate='<b>%{y}</b><br>Fire spots: %{x:,}<extra></extra>'
    ))
    fig.update_layout(**base_layout('Total Fire Spots by State'))
    fig.update_xaxes(title='Fire Spots')
    return fig

def make_stat_cards(states, yr, mo=None):
    df = filter_fires(states, yr, mo)
    
    total_fires = df['firespots'].sum()
    worst_year = df.groupby('year')['firespots'].sum().idxmax()
    worst_year_count = df.groupby('year')['firespots'].sum().max()
    peak_month = df.groupby('month')['firespots'].sum().idxmax()
    peak_month_name = month_names[peak_month]
    worst_state = df.groupby('state')['firespots'].sum().idxmax()

    cards = [
        ('Total Fire Spots', f'{total_fires:,}', '1999 - 2019'),
        ('Worst Year', str(worst_year), f'{worst_year_count:,} fire spots'),
        ('Peak Month', peak_month_name, 'highest avg fire activity'),
        ('Most Affected State', worst_state.title(), 'highest total fire spots'),
    ]

    return cards

def make_fig_map(states, yr, mo=None):
    df = filter_fires(states, yr, mo)
    
    # Aggregate by state
    agg = df.groupby('state')['firespots'].sum().reset_index()
    
    # Map full state names back to abbreviations 
    reverse_abbr = {v: k for k, v in state_abbr.items()}
    agg['sigla'] = agg['state'].map(reverse_abbr)
    agg['state_title'] = agg['state'].str.title()
    
    fig = go.Figure(go.Choropleth(
        geojson=geojson,
        locations=agg['sigla'],
        z=agg['firespots'],
        featureidkey='properties.sigla',
        colorscale='YlOrRd',
        marker_line_color='white',
        marker_line_width=0.5,
        colorbar=dict(
            title=dict(text='Fire Spots', font=dict(color='white')),
            tickfont=dict(color='white'),
        ),
        hovertemplate='<b>%{customdata}</b><br>Fire Spots: %{z:,}<extra></extra>',
        customdata=agg['state_title'],
    ))
    
    fig.update_geos(
        fitbounds='locations',
        visible=False,
        bgcolor=COLOURS['bg'],
    )
    
    fig.update_layout(
    **base_layout('Total Fire Spots by State'),
    geo=dict(
        bgcolor=COLOURS['bg'],
        lakecolor=COLOURS['bg'],
        landcolor=COLOURS['panel'],
    ),
    height=400,
)
    return fig

print("make_fig_map ready ✓")
print("All 6 figures built successfully:")
print("  make_fig_annual →", type(make_fig_annual(None, INIT_YR)))
print("  make_fig_seasonal →", type(make_fig_seasonal(None, INIT_YR)))
print("  make_fig_heatmap →", type(make_fig_heatmap(INIT_YR)))
print("  make_fig_def_state →", type(make_fig_def_state(None, INIT_YR)))
print("  make_fig_state_bar →", type(make_fig_state_bar(INIT_YR)))

make_fig_map ready ✓
All 6 figures built successfully:
  make_fig_annual → <class 'plotly.graph_objs._figure.Figure'>
  make_fig_seasonal → <class 'plotly.graph_objs._figure.Figure'>
  make_fig_heatmap → <class 'plotly.graph_objs._figure.Figure'>
  make_fig_def_state → <class 'plotly.graph_objs._figure.Figure'>
  make_fig_state_bar → <class 'plotly.graph_objs._figure.Figure'>


### App Layout

In [7]:
app = JupyterDash(__name__, external_stylesheets=[
    'https://fonts.googleapis.com/css2?family=Oswald:wght@400;600;700&display=swap',
    'https://cdnjs.cloudflare.com/ajax/libs/rc-slider/9.7.1/assets/index.min.css'
])

slider_style = {
    'color': 'white',
    'accentColor': 'white',
}

ACCENT = COLOURS['accent1']

app.layout = html.Div(style={'backgroundColor': COLOURS['bg'], 'minHeight': '100vh',
                              'padding': '18px 22px', 'fontFamily': FONT}, children=[

    # Header
    html.Div([
    html.Div([
        html.H1('𖠰 Amazon Fires & Deforestation 𖠰',
                style={'color': COLOURS['Title'], 'margin': '0', 'fontSize': '30px', 'fontWeight': '700'}),
        html.P('Fire activity, deforestation, and ENSO climate patterns in the Brazilian Amazon 1999–2019',
               style={'color': COLOURS['neutral'], 'margin': '4px 0 0', 'fontSize': '18px'}),
    ], style={**card_style, 'display': 'inline-block', 'padding': '24px',
}),
], style={'textAlign': 'center', 'marginBottom': '16px'}),

    intro_panel, 
    
    # ENSO info
    enso_info,

    # Fire spot explanation
    firespot_info,
        
    # Controls
    html.Div(style={**card_style, 'marginBottom': '14px', 'padding': '14px 18px'}, children=[
        html.Div(style={'display':'flex','gap':'30px','flexWrap':'wrap','alignItems':'flex-end'}, children=[
            html.Div(style={'flex':'3','minWidth':'260px'}, children=[
                html.Label('Year Range', style=label_style),
                dcc.RangeSlider(id='year-slider', min=YEAR_MIN, max=YEAR_MAX, step=1,
                    value=[YEAR_MIN, YEAR_MAX],
                    marks={y: {'label': str(y), 'style': {'color': COLOURS['neutral'], 'fontSize': '15px'}}
                           for y in range(YEAR_MIN, YEAR_MAX+1, 2)},
                    tooltip={'placement':'bottom','always_visible':False},
                    className='white-slider'),
                
            ]),
            html.Div(style={'flex':'2','minWidth':'200px'}, children=[
                html.Label('Filter States', style=label_style),
                dcc.Dropdown(id='state-dropdown',
                    options=[{'label': s.title(), 'value': s} for s in ALL_STATES],
                    multi=True, placeholder='All states…', style={'fontSize':'15px'}),
            ]),
        ]),
    ]),

    # Row 1
    html.Div(style={'display':'flex','gap':'12px','marginBottom':'12px'}, children=[
        html.Div(style={**card_style,'flex':'3'}, children=[
            dcc.Graph(id='fig-annual', figure=make_fig_annual(None, INIT_YR), config={'displayModeBar':False})]),
        html.Div(style={**card_style,'flex':'2'}, children=[
            dcc.Graph(id='fig-seasonal', figure=make_fig_seasonal(None, INIT_YR), config={'displayModeBar':False})]),
    ]),

    # Row 2
    html.Div(style={'display':'flex','gap':'15px','marginBottom':'15px'}, children=[
        html.Div(style={**card_style,'flex':'3'}, children=[
            dcc.Graph(id='fig-heatmap', figure=make_fig_heatmap(INIT_YR), config={'displayModeBar':False})]),
        html.Div(style={**card_style,'flex':'2'}, children=[
            dcc.Graph(id='fig-state-bar', figure=make_fig_state_bar(INIT_YR), config={'displayModeBar':False})]),
    ]),

    hotspot_info,

    # Map row
    html.Div(style={'display':'flex','gap':'12px','marginBottom':'12px'}, children=[
    html.Div(style={**card_style, 'flex':'1'}, children=[
        dcc.Graph(id='fig-map', figure=make_fig_map(None, INIT_YR), config={'displayModeBar':False})
    ]),
]),

    # Stat cards
    html.Div(id='stat-cards', children=make_stat_panel(None, INIT_YR)),

    
    # Row 3
    html.Div(style={'display':'flex','gap':'12px','marginBottom':'12px'}, children=[
        html.Div(style={**card_style,'flex':'3'}, children=[
            dcc.Graph(id='fig-def-state', figure=make_fig_def_state(None, INIT_YR), config={'displayModeBar':False})]),
    ]), 
    
    deforestation_panel,
    
    # Footer
    html.Div(style={**card_style, 'borderLeft': '4px solid ' + ACCENT, 'padding': '12px 16px'}, children=[
    html.Label('Summary: ', style={
        **label_style, 'fontSize': '16px', 'marginBottom': '8px'
    }),
    html.P('Fire activity peaks July through October and spikes during El Niño years  '
       '2002 was the worst year on record with 273,873 fire spots. '
       'Deforestation peaked in 2004 at 27,772 km², then fell sharply. '
       'Deforestation rebounded from 2015 onward, closely tracking the resurgence in fire activity. '
       'Mato Grosso and Pará consistently dominate both fire spots and deforestation, sitting on the arc of deforestation. '
       'Amazonas, historically low in deforestation, has seen a steady rise since 2015  '
       'growing from 712 km² to 1,421 km² by 2019.',
       style={'color': COLOURS['neutral'], 'margin': '1', 'fontSize': '15px', 'lineHeight': '1.7'}),
]),


    html.Div(style={'display': 'flex', 'gap': '14px', 'marginBottom': '14px'}, children=[
    html.Div(style={'flex': '3'}, children=[dataset_panel]),
    html.Div(style={'flex': '1'}, children=[author_panel]),
]),

])

### Call Backs

In [8]:
@app.callback(
    Output('stat-cards',    'children'),
    Output('fig-annual',    'figure'),
    Output('fig-seasonal',  'figure'),
    Output('fig-heatmap',   'figure'),
    Output('fig-def-state', 'figure'),
    Output('fig-state-bar', 'figure'),
    Output('fig-map',       'figure'),
    Input('year-slider',    'value'),
    Input('state-dropdown', 'value'),
)
def update_all(year_range, selected_states):
    yr  = year_range      if year_range      else [YEAR_MIN, YEAR_MAX]
    sts = selected_states if selected_states else None
    return (
        make_stat_panel(sts, yr),
        make_fig_annual(sts, yr),
        make_fig_seasonal(sts, yr),
        make_fig_heatmap(yr),
        make_fig_def_state(sts, yr),
        make_fig_state_bar(yr),
        make_fig_map(sts, yr),
    )

### Run App

In [9]:
if __name__ == '__main__':
    app.run(jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/


In [10]:
app.run(mode='inline', height=1600, debug=False)